# Time Derivatives Demo

This notebook shows how to evaluate acceleration together with jerk, snap, and crackle using the analytic `solidfmm` time-derivative path.

In [ ]:
import jax
import jax.numpy as jnp

from jaccpot import FastMultipoleMethod, FMMPreset


In [ ]:
key = jax.random.PRNGKey(0)
key_pos, key_mass, key_vel = jax.random.split(key, 3)
n = 512

positions = jax.random.uniform(
    key_pos,
    (n, 3),
    minval=-1.0,
    maxval=1.0,
    dtype=jnp.float32,
)
masses = jax.random.uniform(
    key_mass,
    (n,),
    minval=0.5,
    maxval=1.5,
    dtype=jnp.float32,
)
velocities = jax.random.uniform(
    key_vel,
    (n, 3),
    minval=-0.2,
    maxval=0.2,
    dtype=jnp.float32,
)


In [ ]:
solver = FastMultipoleMethod(
    preset=FMMPreset.FAST,
    basis="solidfmm",
)

accelerations, (jerk, snap, crackle) = solver.compute_accelerations_with_time_derivatives(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    max_time_derivative_order=3,
    mode="accurate",
)


In [ ]:
print("acc shape:", accelerations.shape)
print("jerk shape:", jerk.shape)
print("snap shape:", snap.shape)
print("crackle shape:", crackle.shape)

print("acc mean norm:", jnp.mean(jnp.linalg.norm(accelerations, axis=1)))
print("jerk mean norm:", jnp.mean(jnp.linalg.norm(jerk, axis=1)))
print("snap mean norm:", jnp.mean(jnp.linalg.norm(snap, axis=1)))
print("crackle mean norm:", jnp.mean(jnp.linalg.norm(crackle, axis=1)))


## Prepared-State Reuse

For repeated evaluations on the same particle configuration, prepare the state once and evaluate the time derivatives multiple times for different velocities or target subsets.

In [ ]:
state = solver.prepare_state(
    positions,
    masses,
    leaf_size=16,
    max_order=4,
)

target_indices = jnp.arange(0, 32, dtype=jnp.int32)
acc_subset, (jerk_subset, snap_subset) = solver.evaluate_prepared_state_with_time_derivatives(
    state,
    velocities,
    target_indices=target_indices,
    max_time_derivative_order=2,
    mode="accurate",
)

print("subset acc shape:", acc_subset.shape)
print("subset jerk shape:", jerk_subset.shape)
print("subset snap shape:", snap_subset.shape)
